# 🔍 VC Analyst — AI Startup Evaluator

Venture-scale AI startup evaluation powered by xAI Grok — running in Google Colab.

---

## Before You Start

Add your API keys via the **🔑 Secrets** icon in the left sidebar:

| Secret | Required? | Where to get it |
|--------|-----------|------------------|
| `XAI_API_KEY` | ✅ Required | [console.x.ai](https://console.x.ai) |
| `ANTHROPIC_API_KEY` | Optional (fallback LLM) | [console.anthropic.com](https://console.anthropic.com) |
| `LOGFIRE_TOKEN` | Optional (trace viewer) | [logfire.pydantic.dev](https://logfire.pydantic.dev) — free account |

Then: **Runtime → Run all** and click the Gradio public link that appears after Cell 6.

---

### What Each Cell Does
| Cell | Action |
|------|--------|
| 2 | Clone / update the repo |
| 3 | Install core dependencies |
| 4 | Configure API keys from Colab Secrets |
| 5 | *(Optional)* Enable Logfire traces — run before Cell 6 |
| 6 | **Launch the app** — Gradio public link appears here |
| 7 | *(Optional)* Enable Browser Research for richer URL data |

In [ ]:
# ─── Cell 2: Clone / Update Repo ─────────────────────────────────────────────
import os

REPO_URL = "https://github.com/AS230924/Agents.git"
BRANCH   = "vc-analyst-agent"
REPO_DIR = "/content/Agents"

if os.path.exists(REPO_DIR):
    print("Repo exists — pulling latest changes...")
    os.system(f'git -C "{REPO_DIR}" pull origin {BRANCH} --rebase')
else:
    print("Cloning repo...")
    os.system(f'git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}')

print("✅ Repo ready")

In [ ]:
# ─── Cell 3: Install Core Dependencies ───────────────────────────────────────
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "openai", "anthropic", "httpx", "beautifulsoup4",
    "gradio", "pydantic", "python-dotenv"
])
print("✅ Core dependencies installed")

In [ ]:
# ─── Cell 4: Configure API Keys ──────────────────────────────────────────────
import os
from google.colab import userdata

# Required
os.environ["XAI_API_KEY"]  = userdata.get("XAI_API_KEY")

# Model & Gradio config
os.environ["GROK_MODEL"]   = "grok-4-1-fast-reasoning"
os.environ["GRADIO_SHARE"] = "1"   # Required for a public Gradio link in Colab

# Optional keys — silently skip if not set
for _key in ("ANTHROPIC_API_KEY",):
    try:
        os.environ[_key] = userdata.get(_key)
    except Exception:
        pass

print("✅ Environment configured")
print(f"   XAI key set:       {bool(os.environ.get('XAI_API_KEY'))}")
print(f"   Anthropic key set: {bool(os.environ.get('ANTHROPIC_API_KEY'))}")
print(f"   Model:             {os.environ['GROK_MODEL']}")

In [ ]:
# ─── Cell 5 (OPTIONAL): Enable Logfire Traces ─────────────────────────────────
# Run this cell BEFORE Cell 6 to trace every LLM call in a cloud dashboard.
#
# Setup (one-time):
#   1. Create a free account at https://logfire.pydantic.dev
#   2. Copy your token from the dashboard
#   3. Add it as LOGFIRE_TOKEN in Colab Secrets (🔑 icon in left sidebar)
#   4. Run this cell, then run Cell 6
#
# View traces at: https://logfire.pydantic.dev  (free tier: 10M spans/month)

import subprocess, sys, os
from google.colab import userdata

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "logfire[openai,anthropic]"])

import logfire

os.environ["LOGFIRE_TOKEN"] = userdata.get("LOGFIRE_TOKEN")
logfire.configure(token=os.environ["LOGFIRE_TOKEN"])

logfire.instrument_openai()      # auto-traces all OpenAI SDK calls (Grok uses this too)
logfire.instrument_anthropic()   # auto-traces all Anthropic SDK calls

print("✅ Logfire tracing active")
print("📊 View traces at: https://logfire.pydantic.dev")

In [ ]:
# ─── Cell 6: Launch Gradio App ────────────────────────────────────────────────
import sys
sys.path.insert(0, "/content/Agents/VC Analyst")

from vc_analyst.gradio_app import launch
launch()
# ↓ Public Gradio link appears here (e.g. https://xxxx.gradio.live)
# If Logfire is enabled (Cell 5), go to logfire.pydantic.dev to see LLM traces

## ⚙️ Optional: Browser Research

Enables richer data extraction from startup URLs using Playwright + DuckDuckGo search.

**Note:** Takes ~60 seconds to download Chromium. Run this cell, then re-run Cell 6.

In [ ]:
# ─── Cell 7 (OPTIONAL): Enable Browser Research ───────────────────────────────
# Uncomment all lines below and run, then re-run Cell 6.

# import subprocess, sys, os
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "playwright", "duckduckgo-search"])
# subprocess.check_call(["playwright", "install", "chromium"])
# os.environ["USE_BROWSER_RESEARCH"] = "1"
# print("✅ Browser Research enabled — re-run Cell 6")